In [1]:
import pandas as pd
from rdkit.Chem import MolFromSmiles
from torch_geometric.data import Data
import torch
from rdkit.Chem.rdForceFieldHelpers import MMFFOptimizeMolecule
from rdkit.Chem import AddHs, AllChem

In [2]:
data_raw = pd.read_csv(r'C:\Users\AI1-PC\PycharmProjects\main\datasets\polymers\OPV_poly_exp_all_data.csv', sep=',', index_col=None)
# data_no_dub = data_raw.drop_duplicates(subset=['SMILES'])

In [4]:
data_raw.describe()

,GAP,HOMO,LUMO
count,4130.000000,4130.000000,4130.000000
mean,2.263615,5.508702,3.215010
std,1.199828,0.613115,0.828803
min,0.000000,2.780000,0.000000
25%,1.660000,5.220000,3.020000
50%,1.840000,5.400000,3.500000
75%,2.267500,5.610000,3.720000
max,10.020000,10.360000,4.900000


In [3]:
homo_array = data_raw['HOMO'].to_numpy()
lumo_array = data_raw['LUMO'].to_numpy()
gap_array = data_raw['GAP'].to_numpy()
smiles_array = data_raw['SMILES'].to_numpy()

In [4]:
def calc_data(smiles, gap, homo, lumo):
    mol = MolFromSmiles(smiles)
    molH = AddHs(mol)
    AllChem.EmbedMolecule(molH, useRandomCoords=True, maxAttempts=100)
    AllChem.MMFFOptimizeMolecule(molH, mmffVariant='MMFF94s', maxIters=100)

    x = []
    pos = []
    for i, atom in enumerate(molH.GetAtoms()):
        x.append(atom.GetAtomicNum())
        positions = molH.GetConformer().GetAtomPosition(i)
        pos.append([positions.x, positions.y, positions.z])

    x = torch.tensor(x, dtype=torch.long)
    pos = torch.tensor(pos, dtype=torch.float32)
    y = torch.tensor([[gap, homo, lumo]], dtype=torch.float32)

    data = Data(
        x=x,
        pos=pos,
        y=y
    )

    return data

In [ ]:
data_list = []
for s, y0, y1, y2 in zip(smiles_array, gap_array, homo_array, lumo_array):
    data = calc_data(s, y0, y1, y2)
    data_list.append(data)
    print(len(data_list))

In [ ]:
data_list

In [7]:
for i in range(len(data_list)):
    torch.save(data_list[i], fr'C:\Users\AI1-PC\PycharmProjects\main\datasets\polymers\OPV_poly_exp_all_data_graph_geom_to_prop_Hs_dimenet_v_1\{i}.pt')